# 评估模块（价格预测）

本 notebook 提供统一的评估函数与简易回测框架：

| 函数 | 说明 |
|------|------|
| `regression_metrics(y_true, y_pred)` | 回归指标：RMSE / MAE / R² |
| `direction_accuracy(y_true, y_pred, y_current)` | 方向准确率（涨跌） |
| `print_comparison_table(results)` | 多模型对比表格 |
| `simple_backtest(y_true, y_pred, y_current)` | 基于预测方向的简易 PnL 回测 |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """计算回归评估指标：RMSE / MAE / R²"""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"RMSE": rmse, "MAE": mae, "R2": r2}


def direction_accuracy(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_current: np.ndarray,
) -> float:
    """方向准确率：预测的涨跌方向与实际是否一致。

    y_true:    未来真实价格
    y_pred:    模型预测的未来价格
    y_current: 当前时刻价格
    """
    true_dir = np.sign(y_true - y_current)
    pred_dir = np.sign(y_pred - y_current)
    return np.mean(true_dir == pred_dir)


def print_comparison_table(results: list):
    """打印多模型对比表。

    results: list of dict, 每个 dict 包含 'model', 'RMSE', 'MAE', 'R2',
             'train_time', 'infer_time' 等字段。
    """
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    return df


def simple_backtest(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_current: np.ndarray,
) -> dict:
    """基于预测方向的简易 PnL 回测。

    策略：若预测涨则买入（+1），预测跌则卖出（-1），否则持平（0）。
    PnL = sum(position * (y_true - y_current))。
    """
    position = np.sign(y_pred - y_current)
    pnl_per_step = position * (y_true - y_current)
    cumulative_pnl = np.cumsum(pnl_per_step)
    return {
        "total_pnl": float(cumulative_pnl[-1]),
        "mean_pnl": float(np.mean(pnl_per_step)),
        "sharpe": float(np.mean(pnl_per_step) / (np.std(pnl_per_step) + 1e-9) * np.sqrt(252 * 360)),
        "win_rate": float(np.mean(pnl_per_step > 0)),
        "cumulative_pnl": cumulative_pnl,
    }


print("evaluation 模块已定义：regression_metrics / direction_accuracy / print_comparison_table / simple_backtest")